# NanoForge Studio on Colab G4 Blackwell

Select **G4 / RTX PRO 6000 Blackwell** in Colab runtime settings and run the cell. Setup verifies the assigned GPU, runs CUDA kernel tests, repairs an incompatible PyTorch wheel once using pinned PyTorch 2.10.0 + CUDA 12.8, tests dense and MoE checkpoints, then opens the full UI.

All model/training settings stay in the UI. Choose **Mixture of Experts (MoE)** in Architecture to enable experts and top-k routing. Startup is idle and imports no datasets. G4 defaults use Dense Mini, Magic 8k, BF16 compute, TF32, context 512, batch 32 and accumulation 2. Reduce batch size for larger models. All expert weights count toward the <1B limit.

Download checkpoints before ending the runtime; `/content` is temporary. MoE checkpoints require the updated Studio. TPU/XLA does not yet support MoE. The reference MoE dispatch is not a fused/expert-parallel performance implementation.


In [ ]:
from pathlib import Path
import subprocess

# Stop only the Studio process started by an earlier execution of this cell.
if 'nanoforge_studio' in globals():
    nanoforge_studio.stop()

REPO = Path('/content/nanoforge')
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/bananaosint/nanoforge.git',str(REPO)],check=True)
else:
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only'],check=True)

startup = {'__name__': 'nanoforge_colab_ui'}
script = REPO/'scripts/colab_ui.py'
exec(compile(script.read_text(),str(script),'exec'),startup)
nanoforge_studio = startup['launch_studio'](repo=REPO, profile='colab-g4')
startup['show_studio'](nanoforge_studio)


## Session controls

The UI runs in your Colab runtime and controls its G4 Blackwell GPU. The app shows **COLAB STUDIO**, a Colab connection label and the detected device. Local desktop Studio remains separate.

For server diagnostics, run `print(nanoforge_studio.logs())` in another cell. To stop the server safely and save the active training step/checkpoint, run `nanoforge_studio.stop()`. Rerunning the launcher cell does this first; it does not kill unrelated processes. If shutdown needs more time, wait for the checkpoint and call `stop()` again.

If a large download is blocked by the proxy/browser, use Colab's Files panel under `/content/nanoforge-g4-run/checkpoints/`. Copy a `.pt` back into local Studio using **Import .pt checkpoint**. For Drive persistence, mount Drive first and pass an existing Drive folder as `home=` to `launch_studio`; model settings still stay in the UI.

The original cell that starts training immediately is retained in `NanoForge_L4_Headless.ipynb (L4 profile)`.
